# Import libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import round, col
import pyspark
from pyspark.sql.types import StringType, StructType, StructField, IntegerType, DateType,TimestampType,LongType,DecimalType
from pyspark.sql.window import Window


# Spark Optimization parameters

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50485760)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
spark.conf.set("spark.databricks.io.cache.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled","true")



# Read all datasets from URL

In [0]:
 
# akas_url = "https://datasets.imdbws.com/title.akas.tsv.gz"
# titlebasics_url="https://datasets.imdbws.com/title.basics.tsv.gz"
# crew_url="https://datasets.imdbws.com/title.crew.tsv.gz"
# episode_url="https://datasets.imdbws.com/title.episode.tsv.gz"
# principals_url="https://datasets.imdbws.com/title.principals.tsv.gz"
# ratings_url="https://datasets.imdbws.com/title.ratings.tsv.gz"
# namebasics_url="https://datasets.imdbws.com/name.basics.tsv.gz"


# from pyspark import SparkFiles
# spark.sparkContext.addFile(akas_url)
# spark.sparkContext.addFile(titlebasics_url)
# spark.sparkContext.addFile(crew_url)
# spark.sparkContext.addFile(episode_url)
# spark.sparkContext.addFile(principals_url)
# spark.sparkContext.addFile(ratings_url)
# spark.sparkContext.addFile(namebasics_url)

# titleAkasDF = spark.read.csv("file://"+SparkFiles.get("title.akas.tsv.gz"), sep='\t',header=True)
# titleBasicsDF = spark.read.csv("file://"+SparkFiles.get("title.basics.tsv.gz"), sep='\t',header=True)
# titleCrewDF = spark.read.csv("file://"+SparkFiles.get("title.crew.tsv.gz"), sep='\t',header=True)
# titleEpisodeDF = spark.read.csv("file://"+SparkFiles.get("title.episode.tsv.gz"), sep='\t',header=True)
# titlePrincipalsDF = spark.read.csv("file://"+SparkFiles.get("title.principals.tsv.gz"), sep='\t',header=True)
# titleRatingsDF = spark.read.csv("file://"+SparkFiles.get("title.ratings.tsv.gz"), sep='\t',header=True)
# nameBasicsDF = spark.read.csv("file://"+SparkFiles.get("name.basics.tsv.gz"), sep='\t',header=True)



# Connect to Azure Storage Container

In [0]:
# dbutils.fs.unmount("/mnt/imdb_data")

In [0]:
# dbutils.fs.mount(
#   source = "wasbs://input-dir@decasestorage.blob.core.windows.net",
#   mount_point = "/mnt/imdb_data",
#   extra_configs = {"fs.azure.account.key.decasestorage.blob.core.windows.net":"I+w5khI6cqAhv+Nf5FAPNSaidEWGrLHnX9TsyqHLzsiXtP2lcJkFEHkOKjZS53/2ZP4PSQya0AB53DrF41yPjw=="})

Out[11]: True

# Raw Data stored in Bronze Layer

In [0]:
# titleAkasDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titleAkas.parquet")
# titleBasicsDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titleBasics.parquet")
# titleCrewDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titleCrew.parquet")
# titleEpisodeDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titleEpisode.parquet")
# titlePrincipalsDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titlePrincipals.parquet")
# titleRatingsDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/titleRatings.parquet")
# nameBasicsDF.coalesce(1).write.mode('overwrite').parquet("/mnt/imdb_data/nameBasics.parquet")


# Read data for Silver Layer Operations

In [0]:
titleAkaspqDF=spark.read.parquet("/mnt/imdb_data/titleAkas.parquet")
titleBasicspqDF=spark.read.parquet("/mnt/imdb_data/titleBasics.parquet")
titleCrewpqDF=spark.read.parquet("/mnt/imdb_data/titleCrew.parquet")
titleEpisodepqDF=spark.read.parquet("/mnt/imdb_data/titleEpisode.parquet")
titlePrincipalspqDF=spark.read.parquet("/mnt/imdb_data/titlePrincipals.parquet")
titleRatingspqDF=spark.read.parquet("/mnt/imdb_data/titleRatings.parquet")
nameBasicspqDF=spark.read.parquet("/mnt/imdb_data/nameBasics.parquet")


# Persist all Dataframes for iterative operations ahead

In [0]:
#Memory Only operations; can use persist() if scalablity increases

titleRatingspqDF.cache()
titleBasicspqDF.cache()
nameBasicspqDF.cache()
titlePrincipalspqDF.cache()

Out[6]: DataFrame[tconst: string, ordering: string, nconst: string, category: string, job: string, characters: string]

# Fetch all movie data by filtering on titleType

In [0]:
movieDataDF = titleBasicspqDF.join(titleRatingspqDF,titleBasicspqDF.tconst == titleRatingspqDF.tconst)\
.where(col("titleType") == 'movie')\
.select(titleBasicspqDF.tconst,"titleType","primaryTitle","originalTitle","isAdult","startYear","endYear","runtimeMinutes","genres","averageRating","numVotes")


# Get the top 10 movies with minimum 50 votes and the ranking should be based on (numVotes/averageNumberOfVotes)* averageRating

In [0]:
mean = movieDataDF.groupBy().agg(avg(col("numVotes"))).take(1)[0][0]

movieDataDF1=movieDataDF\
.where(col("numVotes") >= 50)\
.withColumn("avg_num_Votes", lit(mean)).select("*",round("avg_num_Votes",2).alias("average_numVotes")).drop("avg_num_Votes")

movieDataDF2=movieDataDF1.withColumn('ranking',((col('numVotes') / col('average_numVotes')) *  col('averageRating')))\
.select("*",round("ranking",2).alias("rank_movie")).drop("ranking","titleType","endYear","primaryTitle")


windowSpec = Window.orderBy(col('rank_movie').desc())
rankedmovieDF = movieDataDF2.withColumn("Ranking", dense_rank().over(windowSpec)).where(col("Ranking") <= 10).drop("rank_movie")


# Save output to delta table "rankedMovies_tbl"

In [0]:
rankedmovieDF.coalesce(1).write.format('delta').mode("overwrite").saveAsTable("rankedMovies_tbl")

# For the above 10 movies, list the persons who are most often credited and list the different titles of the 10 movies.

In [0]:
joined_DF= rankedmovieDF.join(titlePrincipalspqDF,rankedmovieDF.tconst == titlePrincipalspqDF.tconst,'inner')\
.join(nameBasicspqDF,titlePrincipalspqDF.nconst == nameBasicspqDF.nconst,'inner').orderBy("Ranking")

PeopleTitle_DF =joined_DF.select("originalTitle","primaryName")



# Save output to delta table "PeopleTitle_tbl"

In [0]:
PeopleTitle_DF.coalesce(1).write.format('delta').mode("overwrite").saveAsTable("PeopleTitle_tbl")

# Unpersist all cached Dataframes

In [0]:
titleRatingspqDF.unpersist()
titleBasicspqDF.unpersist()
nameBasicspqDF.unpersist()
titlePrincipalspqDF.unpersist()